# Step 2: Filter notebook

In [ ]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point, box, LineString
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from network_connectivity import *

## Imports

#### Import des segments

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "bike"  # walk or bike

if territory == 'GG':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GG/step-1'
    output_step2_path='../../Data/output/GG/step-2'
    output_step3_path='../../Data/output/GG/step-3'


    save_path = '../../Data/output/GG/step-1'

if territory == 'GE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GE/step-1'
    output_step2_path='../../Data/output/GE/step-2'
    output_step3_path='../../Data/output/GE/step-3'


    save_path = '../../Data/output/GE/step-1'

save_filtered_attributes = True

# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if geom is not None and not geom.is_valid else geom)
    print("Geometries cleaned")

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

#### Import des attributs

In [ ]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

attributs_info

**Connectivité du réseau**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'connectivite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

# 1. Ajouter u, v, key si besoin
segmented_net = add_uv_columns(segmented_net)

# 2. Calculer les métriques et le score d’alternatives
segmented_net_index = compute_connectivity_metrics(
    segmented_net,
    buffer_m=35,
    compute_betweenness=False,
    betweenness_k=None,
    crs_meter_epsg=2056,
    main_metrics_only=False,
    conn_index_metric="conn_branching_in_buffer",
)

segmented_net_index['filtered'] = 1


# Preview
segmented_net_index.to_crs(target_crs).head()

In [ ]:
save(save_filtered_attributes, row, segmented_net_index.to_crs(target_crs)[["segment_id", "infra_bike","geometry","conn_branching_in_buffer","conn_index_score","filtered"]], attribute)

In [ ]:
segmented_net_index.to_crs(target_crs)[["key","segment_id", "infra_bike","geometry","conn_branching_in_buffer","conn_index_score","filtered"]]

**Vitesse**

In [ ]:
bike_edges_graph_path = f"{input_file_path}/networkGG/bike_edges_graph.geojson"
bike_edges_graph = gpd.read_file(bike_edges_graph_path)

import json
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

# --- 1) Parse maxspeed -------------------------------------------------------
def parse_maxspeed(v):
    if v is None or pd.isna(v):
        return np.nan
    if isinstance(v, (int, float)):
        return float(v)

    s = str(v).lower().strip()
    if s in {"walk", "walking", "foot"}:
        return 5.0
    if s in {"signals", "none", "variable"}:
        return np.nan

    if "mph" in s:
        m = re.search(r"(\d+)", s)
        return float(m.group(1)) * 1.60934 if m else np.nan

    m = re.search(r"(\d+)", s)
    return float(m.group(1)) if m else np.nan


# --- 2) Highway normalization (list -> str) ----------------------------------
def normalize_highway(v):
    if v is None or pd.isna(v):
        return np.nan
    if isinstance(v, (list, tuple, set)):
        return str(list(v)[0]) if len(v) else np.nan
    return str(v)


# --- 3) Default speeds (CH/urban Europe, adjust if you want) -----------------
DEFAULT_SPEED_BY_HIGHWAY = {
    "motorway": 120, "trunk": 80,
    "primary": 50, "secondary": 50, "tertiary": 50,
    "primary_link": 50, "secondary_link": 50, "tertiary_link": 50,
    "residential": 30, "unclassified": 30,
    "service": 20, "living_street": 20,

    # pedestrian-ish
    "pedestrian": 10,
    "footway": 5, "path": 5, "steps": 5,
    "track": 30,
    "platform": 5, "corridor": 5,
}

ZONE_MAXSPEED_MAP = {
    "CH:urban": 50,
    "CH:rural": 80,
    "CH:motorway": 120,
}


# --- 4) Compute speed_kph on a COPY only ------------------------------------
def compute_speed_kph(df: gpd.GeoDataFrame) -> pd.Series:
    speed = pd.Series(np.nan, index=df.index, dtype="float64")

    # a) use explicit maxspeed tags if present
    for col in ["maxspeed:forward", "maxspeed:backward", "maxspeed"]:
        if col in df.columns:
            speed = speed.fillna(df[col].apply(parse_maxspeed))

    # b) zone-based hints
    if "zone:maxspeed" in df.columns:
        z = df["zone:maxspeed"].astype(str)
        speed = speed.fillna(z.map(ZONE_MAXSPEED_MAP))

    # c) highway fallback
    if "highway" in df.columns:
        hw = df["highway"].apply(normalize_highway)
        speed = speed.fillna(hw.map(DEFAULT_SPEED_BY_HIGHWAY))

    # d) final fallback
    speed = speed.fillna(30.0)

    return speed


# --- 5) Build output gdf and export -----------------------------------------
def export_speed_layer(edges_gdf_graph: gpd.GeoDataFrame, out_dir: Path, basename="edges_speed"):
    out_dir.mkdir(parents=True, exist_ok=True)

    out = edges_gdf_graph.copy()
    out["highway_norm"] = out["highway"].apply(normalize_highway) if "highway" in out.columns else np.nan
    # fill missing highway (optional, conservative)
    out["highway_norm"] = out["highway_norm"].fillna("service")

    out["speed_kph"] = compute_speed_kph(out)

    # keep only useful columns
    keep = [c for c in ["edge_id", "u", "v", "key", "osm_id", "id", "element", "name", "highway", "highway_norm",
                        "maxspeed", "zone:maxspeed", "maxspeed:type", "speed_kph", "length", "len_m", "geometry"]
            if c in out.columns]
    out = out[keep].copy()

    # GeoJSON
    geojson_path = out_dir / f"{basename}.geojson"
    out.to_file(geojson_path, driver="GeoJSON")

    # Parquet (geoparquet)
    parquet_path = out_dir / f"{basename}.parquet"
    out.to_parquet(parquet_path, index=False)

    print("Exported:", geojson_path)
    print("Exported:", parquet_path)
    print("Missing speed_kph:", int(out["speed_kph"].isna().sum()))
    return out



out_dir = Path(f"{input_file_path}/attributs/GG/vitesse")  # choose where you want
speed_gdf = export_speed_layer(bike_edges_graph, out_dir, basename=f"vitesse_all_edges")
